# Step 05 — LLM classification

Step 04 decided *which* buildings are places of activity. This notebook asks,
for each of them, **what activities happen inside and what kind of building it
is**, using an LLM on the evidence step 04 collected.

| | |
|---|---|
| **Reads** | `data/output/04_buildings_enriched.gpkg` — layer `buildings` (39,786 rows, 43 columns) and `building_pois`; `data/output/01_all_pois.gpkg` for the OSM key of each POI's tag; `.env` for the API token (git-ignored, never printed) |
| **Writes** | `data/output/05_llm_input.parquet` — one record per building, the text the model reads, with the model-only columns beside it; `05_llm_plan.parquet` — who is asked; `05_llm_sample_answers.jsonl` — the sample run; the full run's `05_llm_answers.jsonl` is written by `scripts/05_run_llm.py` |
| **Needs** | `pyogrio`, `pandas`, `pyarrow`, `requests` |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **05.1** | **The columns** — which the LLM reads, which only the model needs, which are noise for this step | **implemented** |
| **05.2** | **The prompt input** — one compact record per building from the 15 LLM columns, POI names paired with their OSM tags, sources labelled | **implemented** |
| **05.3** | **The prompt and the output schema** — the system prompt, the label and class lists it must agree with, the JSON schema of an answer, the work rule | **implemented** |
| **05.4** | **Routing** — one call per building where there is evidence, one call per signature where there is only class, land and size | **implemented** |
| **05.5** | **The calls** — a client that validates every reply and re-asks at once, a resumable run with live progress, the ten sample buildings once on the real model | **implemented — sample run; the full run is `scripts/05_run_llm.py`** |
| **05.6** | **Assembly and validation** — answers joined to the buildings, work by rule, signature answers copied, comparison with the rule baseline and the annotated set | pending |

## Why an LLM, and what it is for

The previous pipeline settled this on an annotated set: the LLM reached 78.5 %
against 57.7 % for the rule table, and the whole difference was business-name
world knowledge — the model knows what *Deutsche Bank*, *Ernsting's family* or
*Tischlerei Holzteam* are. This step keeps that design and feeds the model what
step 04 has assembled per building: the POIs on it with names and uses, the
site around it, the cadastre class and name, the OSM footprint tag and name,
the land under it, and its size.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import json
import re
import time
import numpy as np
import pandas as pd
import pyogrio

from config import (
    ENRICHED_BUILDINGS_FILE, ALL_POIS_FILE, OUTPUT_DIR,
    LLM_COLUMN_ROLES, LLM_INPUT_COLS, LLM_MODEL_COLS, LLM_DROPPED_COLS,
    LLM_INPUT_FILE, LLM_RECORD_SAMPLE_PER_GROUP,
    LLM_SYSTEM_PROMPT, LLM_ACTIVITY_LABELS, WORK_IMPLIED_BY,
    LLM_BOSSERHOF_SUBCATEGORIES, LLM_BOSSERHOF_HEADLINES, LLM_BOSSERHOF_CLASSES,
    LLM_CONFIDENCE_LEVELS, LLM_OUTPUT_SCHEMA,
    LLM_PLAN_FILE, LLM_SIGNATURE_AREA_BINS_M2, LLM_SIGNATURE_HEIGHT_BINS_M,
    LLM_API_URL, LLM_MODEL, LLM_MAX_ATTEMPTS,
    LLM_SAMPLE_BUILDING_IDS, LLM_SAMPLE_ANSWERS_FILE, LLM_ANSWERS_FILE, LLM_STATUS_FILE,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols
from lib.llm_record import build_records, GROUP_SAME_USE_FROM, LINE_PREFIXES
from lib.llm_routing import build_plan
from lib.llm_client import read_token, prompt_sha, parse_answer, validate_answer, InvalidAnswer
from lib.llm_run import run, load_answers, valid_answers

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 70)

print('Root :', ROOT_DIR)
print('Input:', ENRICHED_BUILDINGS_FILE.name, '+', ALL_POIS_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input: 04_buildings_enriched.gpkg + 01_all_pois.gpkg


## 1. The columns

The enriched layer carries 43 columns. Not all of them belong in a prompt, and
some must not be there. `LLM_COLUMN_ROLES` in `config.py` assigns every column
one of three roles, with the reason next to it; this section reads the layer,
checks that **every column is assigned** — a column added or removed in step 04
stops this step until it is classified — and shows what each one holds.

**The LLM sees** what describes what happens inside, in three blocks of
falling trust:

1. *what is inside* — `poi_uses`, `poi_names`, `site_uses`, `site_names`
2. *what the building is* — `label_en`, `name`, `osm_tag` (one field from
   `osm_twin_tag` on ALKIS rows and `osm_building` on OSM rows), `osm_twin_name`
3. *where and how big* — `alkis_landuse`, `alkis_landuse_detail` (the parcel's
   coded kind: education and science, health, power plant, campsite ...),
   `osm_landuse`, `city`, `area_m2`, `height_top_max_m`

**Only the model needs** the keys, the weight and the baseline: `building_id`,
`alkis_id`, `ags`, `function`, `volume_3d_m3`, `source`, `n_pois`, `n_sites`,
`address`, `activities`. Two of these are deliberately withheld from the
prompt. `activities` is the rule table's answer from the ALKIS class; shown, it
anchors the model to the rule, hidden, it is the baseline to validate against.
`address` proves a mailbox, not an activity, and the model has no lookup at
call time — asked about an address it would invent a tenant.

**Noise for this step** is step-03 provenance and QA, geometry bookkeeping,
columns already folded into others, the filter's own provenance (`rescued`,
`rescued_by`), and QGIS legend text. They stay in the step 04 output; they do
not enter step 05.

`alkis_landuse_detail` is the newest field, added 2026-09-14 after the size
floor left 6,052 buildings with nothing but class, land use and size. It is
the coded kind of the parcel under the building, one level below
`alkis_landuse`, translated through the AdV codelists in the GDI-DE registry
(see `ALKIS_LANDUSE_DETAIL_EN` in config, with the source URLs). A real case:
`DENIAL01000050Yi`, *Buildings for public purposes*, 787 m², five storeys, an
`office` footprint in OSM, no POI, no name. The class alone reads as
administration; the parcel says *education and science*, and the building
stands beside a state research institute. Filled on 7,493 kept buildings, and
on 821 of the 8,216 that have no POI, site or name — the group it was added for.

The three name columns stay separate on purpose. Measured on the layer: 1,345
buildings have an OSM footprint name that no POI carries, 1,521 an ALKIS name
and nothing else, and 10,466 have POI names but no footprint name. A name
repeated by three sources is a name three sources agree on; merged into one
list the model would not know who said what.

In [2]:
require_file(ENRICHED_BUILDINGS_FILE, 'enriched buildings (step 04)')
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if 'buildings' not in _layers:
    raise AssertionError(f'layer "buildings" missing from {ENRICHED_BUILDINGS_FILE.name}: {_layers}')

print('Reading the enriched buildings (attributes only) ...', flush=True)
t0 = time.perf_counter()
bld = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', read_geometry=False)
print(f'  ok  {len(bld):,} buildings x {len(bld.columns)} columns  [{time.perf_counter() - t0:,.1f}s]')
require_non_empty(bld, 'enriched buildings')
require_unique(bld, 'building_id', 'enriched buildings')

# --- every column must have a role, and every role a column -------------------
_layer_cols = set(bld.columns)
_role_cols = set(LLM_COLUMN_ROLES)
_unassigned = sorted(_layer_cols - _role_cols)
_missing = sorted(_role_cols - _layer_cols)
if _unassigned or _missing:
    raise AssertionError(
        'config.LLM_COLUMN_ROLES and the layer disagree - '
        f'columns in the layer without a role: {_unassigned}; '
        f'roles for columns the layer no longer has: {_missing}. '
        'Classify them in config before running this step.')
print(f'  ok  all {len(_layer_cols)} columns have a role: '
      f'{len(LLM_INPUT_COLS)} the LLM sees, {len(LLM_MODEL_COLS)} model only, {len(LLM_DROPPED_COLS)} dropped here')

# --- what each column holds ------------------------------------------------------
def _example(s):
    v = s.dropna()
    if v.empty:
        return ''
    v = v.iloc[min(7, len(v) - 1)]
    return str(v)[:60]

review = pd.DataFrame({
    'role':      [LLM_COLUMN_ROLES[c][0] for c in bld.columns],
    'filled_%':  (bld.notna().mean() * 100).round(1).to_numpy(),
    'distinct':  bld.nunique().to_numpy(),
    'example':   [_example(bld[c]) for c in bld.columns],
    'why':       [LLM_COLUMN_ROLES[c][1] for c in bld.columns],
}, index=pd.Index(bld.columns, name='column'))
_order = {'llm': 0, 'model': 1, 'drop': 2}
review = review.iloc[np.argsort([_order[r] for r in review['role']], kind='stable')]
for _role, _title in (('llm', 'THE LLM SEES'), ('model', 'ONLY THE MODEL NEEDS'), ('drop', 'NOISE FOR THIS STEP - not read')):
    print()
    print(f'=== {_title} ({int((review["role"] == _role).sum())} columns)')
    print(review.loc[review['role'] == _role, ['filled_%', 'distinct', 'example', 'why']].to_string())

# --- how much of the layer has how much to say --------------------------------------
_has_inside = bld['poi_uses'].notna() | bld['site_uses'].notna()
_has_name = bld['name'].notna() | bld['osm_twin_name'].notna()
_tag = bld['osm_twin_tag'].fillna(bld['osm_building'])
_has_tag = _tag.notna() & (_tag != 'yes')
print()
print('  ..  what the LLM will have to work with, per building:')
print(f'        POIs or a site on it           {int(_has_inside.sum()):>7,}  ({100 * _has_inside.mean():.1f} %)')
print(f'        a name but no POI or site      {int((~_has_inside & _has_name).sum()):>7,}')
print(f'        only an informative OSM tag    {int((~_has_inside & ~_has_name & _has_tag).sum()):>7,}')
print(f'        class, land use and size only  {int((~_has_inside & ~_has_name & ~_has_tag).sum()):>7,}  '
      f'({100 * (~_has_inside & ~_has_name & ~_has_tag).mean():.1f} %) - the per-signature group, section 4')

  ok  04_buildings_enriched.gpkg (42.0 MB)
Reading the enriched buildings (attributes only) ...


  ok  39,786 buildings x 43 columns  [0.5s]
  ok  enriched buildings: 39,786 rows
  ok  enriched buildings.building_id: unique and non-null (39,786)
  ok  all 43 columns have a role: 15 the LLM sees, 10 model only, 18 dropped here

=== THE LLM SEES (15 columns)
                      filled_%  distinct                                             example                                                                                                                               why
column                                                                                                                                                                                                                        
name                      12.0      2528                                           Gärtnerei                                         the cadastre's own label (Tischlerei, Grundschule, Vereinsheim); OSM name on the gap rows
city                     100.0       134                             

## 2. The prompt input

One building, one record, one call. The model runs locally, so tokens cost
nothing but runtime — which is exactly why a record carries only what has
something to say, and says it compactly.

The record is a labelled block, not prose and not JSON. The previous pipeline
sent `key=value` pairs on two lines and reached 78.5 % on the annotated set; this
keeps that shape and changes four things:

* **POI names are paired with their OSM tags** — `Star Tankstelle (amenity=fuel);
  Aral Shop (shop=convenience)`. The building row's `poi_uses` and `poi_names`
  lists are *not* aligned (a building can carry four uses and three names), so
  the pairs come from the `building_pois` layer, ordered by share, and the key
  of each tag is joined from the step 01 POI layer. The key matters: the dry
  run in section 3 showed that a bare `multi` (sport=multi, a multi-sport hall),
  `it` (office=it), `car` (shop=car) or `apartment` (tourism=apartment) means
  nothing to the model, and a school's gym read as a classroom wing. Where many
  POIs share a tag they are grouped under it — a snake farm with 20
  `tourism=attraction` points reads `tourism=attraction x20: Königskobra;
  Netzpython; …` — so no name is lost and the record stays readable.
* **Every line names its source** — `inside`, `site`, `cadastre`, `osm
  footprint`, `land`, `place` — instead of a confidence level. The system prompt
  says how much each source weighs; a name the cadastre gave stays
  distinguishable from one a mapper gave.
* **Empty lines are left out.** Absence is the information. The 1,600 OSM
  gap-fill rows have no register entry, so they have no cadastre line.
* **Numbers are rounded** to whole metres and square metres, and the area has
  no thousands separator — `24,395 m2` reads as a decimal to a German reader.

Three records as they will be sent. There is no id line: one building per call,
the answer is joined by position, and an id is only a string the model might
read something into. The `building_id` sits beside the record in the file.

```
cadastre: Buildings for public purposes
osm footprint: office
land: public facilities, education and science | osm: residential
place: Braunschweig | footprint 787 m2 | height 17 m
```

```
inside: Peeck Freie Tankstelle Ehmen (amenity=fuel)
site: Peeck Freie Tankstelle Ehmen (amenity=fuel)
cadastre: Gas station
osm footprint: yes
land: industry and manufacturing | osm: commercial
place: Wolfsburg | footprint 245 m2 | height 4 m
```

```
inside: unnamed (sport=multi)
site: Wilhelm-Gymnasium Abt. Leonhardstraße (amenity=school)
cadastre: General education school, named "Schule Leonhardstraße"
osm footprint: yes
land: public facilities, education and science
place: Braunschweig | footprint 675 m2 | height 12 m
```

The rendering lives in `lib/llm_record.py`, importable on its own, so the
validation in 05.5 can reproduce exactly what the model saw. Names lose line
breaks and semicolons before rendering, and every line of every record must
start with one of the six labels — a check below enforces it. The output file
carries each record with the model-only columns beside it, and an `evidence`
column naming the group the building falls in: `poi_or_site`, `name_only`,
`tag_only`, `class_only`. Section 4 routes the last group per signature.

In [3]:
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if 'building_pois' not in _layers:
    raise AssertionError(f'layer "building_pois" missing from {ENRICHED_BUILDINGS_FILE.name}: {_layers}')
pairs = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='building_pois', read_geometry=False)
require_cols(pairs, ['building_id', 'poi_id', 'poi_role', 'poi_use', 'name', 'share_in_building', 'share_of_site'], 'building_pois')
_orphans = set(pairs['building_id']) - set(bld['building_id'])
if _orphans:
    raise AssertionError(f'{len(_orphans):,} building_pois rows point at buildings not in the layer, e.g. {sorted(_orphans)[:3]}')
print(f'  ok  {len(pairs):,} building-POI pairs on {pairs["building_id"].nunique():,} buildings '
      f'({int((pairs["poi_role"] != "site").sum()):,} own POIs, {int((pairs["poi_role"] == "site").sum()):,} site memberships)')

# --- the OSM key of each POI's use, from step 01 ------------------------------------
require_file(ALL_POIS_FILE, 'POIs (step 01)')
_keys = pyogrio.read_dataframe(ALL_POIS_FILE, layer='pois', read_geometry=False, columns=['poi_id', 'poi_use', 'poi_use_tag'])
require_unique(_keys, 'poi_id', 'POIs')
pairs = pairs.merge(_keys.rename(columns={'poi_use': '_poi_use_01'}), on='poi_id', how='left')
if pairs['poi_use_tag'].isna().any():
    raise AssertionError(f'{int(pairs["poi_use_tag"].isna().sum()):,} building_pois rows have no POI in {ALL_POIS_FILE.name}')
if (pairs['poi_use'] != pairs['_poi_use_01']).any():
    raise AssertionError('poi_use differs between step 01 and step 04 for the same poi_id - the two files are from different runs')
pairs = pairs.drop(columns='_poi_use_01')
print(f'  ok  every pair has its OSM key; {pairs["poi_use_tag"].nunique()} keys, most frequent: '
      + ', '.join(f'{k} {n:,}' for k, n in pairs['poi_use_tag'].value_counts().head(5).items()))

print('Rendering one record per building ...', flush=True)
t0 = time.perf_counter()
records = build_records(bld[list(LLM_INPUT_COLS) + ['building_id', 'source']], pairs)
print(f'  ok  {len(records):,} records  [{time.perf_counter() - t0:,.1f}s]')
require_unique(records, 'building_id', 'records')
if (records['n_chars'] < 40).any():
    raise AssertionError('a record came out nearly empty - every building has at least land and size')
_stray = [l for rec in records['record'] for l in rec.split('\n') if not l.startswith(LINE_PREFIXES)]
if _stray:
    raise AssertionError(f'{len(_stray)} record lines start with no source label, e.g. {_stray[:3]!r}')
print(f'  ok  every line of every record starts with one of {LINE_PREFIXES}')

# --- what the model will read: size and shape --------------------------------------
print()
print('  ..  record length (characters):')
print(records['n_chars'].describe(percentiles=[.5, .9, .99]).round(0).astype(int).to_string())
_longest = records.sort_values('n_chars', ascending=False).head(3)
print('  ..  the three longest records belong to: '
      + ', '.join(f'{r.building_id} ({r.n_chars:,} chars)' for r in _longest.itertuples()))
print()
print('  ..  by evidence group:')
_g = records.groupby('evidence').agg(buildings=('building_id', 'size'), median_chars=('n_chars', 'median'))
print(_g.reindex(['poi_or_site', 'name_only', 'tag_only', 'class_only']).to_string())

# --- read some, one group at a time -----------------------------------------------------
_rng = np.random.default_rng(7)
for _grp in ('poi_or_site', 'name_only', 'tag_only', 'class_only'):
    _pool = records.index[records['evidence'] == _grp]
    print()
    print(f'=== {_grp} ({len(_pool):,} buildings) - {LLM_RECORD_SAMPLE_PER_GROUP} at random:')
    for _i in _rng.choice(_pool, size=min(LLM_RECORD_SAMPLE_PER_GROUP, len(_pool)), replace=False):
        print(records.at[_i, 'record'])
        print('-' * 60)
print()
print(f'=== the longest record, to see how grouping (same tag x{GROUP_SAME_USE_FROM}+) keeps it readable:')
print(records.at[_longest.index[0], 'record'])

# --- write: the record next to the model-only columns ----------------------------------
llm_input = bld[list(LLM_MODEL_COLS)].merge(records, on='building_id', how='left')
if llm_input['record'].isna().any():
    raise AssertionError('a building has no record after the merge')
llm_input = llm_input[['building_id', 'evidence', 'record', 'n_chars'] + [c for c in LLM_MODEL_COLS if c != 'building_id']]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
llm_input.to_parquet(LLM_INPUT_FILE, index=False)
_back = pd.read_parquet(LLM_INPUT_FILE)
if len(_back) != len(bld) or list(_back.columns) != list(llm_input.columns):
    raise AssertionError('the parquet file does not read back as written')
print()
print(f'  ok  {len(_back):,} rows x {len(_back.columns)} columns -> {LLM_INPUT_FILE.name} '
      f'({LLM_INPUT_FILE.stat().st_size / 1e6:.1f} MB); columns: {list(_back.columns)}')

  ok  building_pois: has ['building_id', 'poi_id', 'poi_role', 'poi_use', 'name', 'share_in_building', 'share_of_site']
  ok  41,285 building-POI pairs on 30,103 buildings (22,633 own POIs, 18,652 site memberships)
  ok  01_all_pois.gpkg (14.2 MB)
  ok  POIs.poi_id: unique and non-null (27,843)
  ok  every pair has its OSM key; 24 keys, most frequent: landuse 14,083, amenity 11,086, shop 6,288, office 1,709, building 1,539
Rendering one record per building ...


  ok  39,786 records  [112.3s]
  ok  records.building_id: unique and non-null (39,786)
  ok  every line of every record starts with one of ('inside: ', 'site: ', 'cadastre: ', 'osm footprint: ', 'land: ', 'place: ')

  ..  record length (characters):
count    39786
mean       208
std         55
min         87
50%        200
90%        270
99%        375
max       3094
  ..  the three longest records belong to: DENIAL0100005xF6 (3,094 chars), DENIAL9100005MxH (1,689 chars), DENIAL01000051gc (1,539 chars)

  ..  by evidence group:
             buildings  median_chars
evidence                            
poi_or_site      30103         212.0
name_only         1467         182.0
tag_only          2164         169.0
class_only        6052         158.0

=== poi_or_site (30,103 buildings) - 3 at random:
inside: KK Schießstand Almke (tourism=museum)
cadastre: Buildings for business or commerce
land: forestry
place: Wolfsburg | footprint 30 m2 | height 2 m
--------------------------------------


  ok  39,786 rows x 13 columns -> 05_llm_input.parquet (2.9 MB); columns: ['building_id', 'evidence', 'record', 'n_chars', 'alkis_id', 'ags', 'function', 'volume_3d_m3', 'source', 'n_pois', 'n_sites', 'address', 'activities']


## 3. The prompt and the output schema

The system prompt is the previous pipeline's, which reached 78.5 % on the
annotated set. What worked is kept verbatim: the twelve label definitions with
their core ideas, and the Bosserhof catalogue. What changed, and why:

* **The input.** The old prompt described two fields, `precise_known_info` and
  `general_building_context`. The new one describes the six record lines and
  how much each weighs: the inside line is the occupants and the strongest
  evidence; names carry world knowledge; the cadastre and the footprint say what
  kind of building it is; land and size are context, and size says how large an
  activity is, never which.
* **Nothing about scope.** Every record is a building with activity, so the old
  enterable-building rule and the empty answer are gone, and with them every
  sentence the model would have had to weigh for nothing.
* **No examples.** Definitions only, so the model generalises from meaning
  instead of matching the example it was shown.
* **Work is defined as the MiD trip purpose** — people come because they are
  employed here — and the old paragraph demanding work next to every visitor
  label is gone. That paragraph asked the model to decide for each building
  whether staff count, and it decided differently each time. The staff of a
  shop or a school are a fact, not a judgement: `WORK_IMPLIED_BY` in config adds
  work by rule to every building that has any other label, after the model. The
  model's own work stays visible (`work_from` in 05.5), because it separates a
  place of production or office work from a shop that happens to employ people.
* **Bosserhof by understanding, not by clue.** Form a picture of the place from
  the whole record, read what each category means, choose the one that fits.
  A subcategory only when the record favours it over its siblings, otherwise the
  headline; the closest class when nothing fits well. The old source ordering
  and dominance heuristic, which made the register class the primary clue, are
  gone — forty shops and 24,000 m² are a shopping centre whatever the class says.
* **Confidence and a shorter reason.** Three disjoint tiers, and 120 words
  instead of 400: on the local model every output token is runtime.

**Dry run before any real call.** Ten real records — every evidence group plus
the mall, a fire station kept by its ALKIS name, a hotel and a hall known only
from their site, a church, a supermarket with tenants — each classified three
times by an independent stand-in that read the prompt and one record cold.
Labels were identical in 8 of 10 (ignoring work), confidence in 9 of 10, the
Bosserhof class in 5 of 10; the mall came back `shopping centers` three of
three. Every Bosserhof split traced to a test the wording left open — the
medium and low tiers described the same record, Part B step 3 gave two rules
that pulled apart, a heading word matched a word in the record, a site set the
kind of building — and each was closed with one sentence. The same run found
what the renderer had hidden from the model, fixed in section 2. The stand-ins
were a stronger model than the target, so the agreement is an upper bound;
05.5 repeats the test on the real model.

The prompt lives in `config.py` as `LLM_SYSTEM_PROMPT`, next to the lists an
answer is validated against: `LLM_ACTIVITY_LABELS`, `LLM_BOSSERHOF_CLASSES`,
`LLM_CONFIDENCE_LEVELS`, and the JSON schema `LLM_OUTPUT_SCHEMA`. The cell below
prints the prompt as the model receives it and checks that the prompt text and
the lists agree — a label or class edited in one place and not the other stops
the notebook here.

In [4]:
print(LLM_SYSTEM_PROMPT)
print('=' * 78)

# --- the prompt and the validation lists must agree --------------------------------
_lab_block = LLM_SYSTEM_PROMPT.split('ALLOWED ACTIVITY LABELS', 1)[1].split('PROCEDURE:', 1)[0]
_labels_in_prompt = tuple(l[2:].strip() for l in _lab_block.splitlines() if l.startswith('- '))
if _labels_in_prompt != LLM_ACTIVITY_LABELS:
    raise AssertionError(f'activity labels differ - prompt {_labels_in_prompt} vs config {LLM_ACTIVITY_LABELS}')
for _lab in LLM_ACTIVITY_LABELS:
    if f'\n- {_lab}\n' not in LLM_SYSTEM_PROMPT.split('LABEL DEFINITIONS', 1)[1]:
        raise AssertionError(f'label "{_lab}" has no definition block in the prompt')

_cat_block = LLM_SYSTEM_PROMPT.split('BOSSERHOF CATEGORIES AND WHAT THEY MEAN', 1)[1].split('OUTPUT FORMAT', 1)[0]
_prompt_cats, _head = {}, None
for _l in _cat_block.splitlines():
    _m = re.match(r'^(\d)\) (.+)$', _l)
    if _m:
        _head = _m.group(2).strip()
        _prompt_cats[_head] = []
    elif _l.startswith('- ') and not _l.startswith('- no fixed subcategories'):
        _prompt_cats[_head].append(_l[2:].strip())
_prompt_cats = {k: tuple(v) for k, v in _prompt_cats.items()}
if _prompt_cats != LLM_BOSSERHOF_SUBCATEGORIES:
    _diff = {k: (_prompt_cats.get(k), LLM_BOSSERHOF_SUBCATEGORIES.get(k))
             for k in set(_prompt_cats) | set(LLM_BOSSERHOF_SUBCATEGORIES) if _prompt_cats.get(k) != LLM_BOSSERHOF_SUBCATEGORIES.get(k)}
    raise AssertionError(f'Bosserhof catalogue differs between prompt and config: {_diff}')
if len(set(LLM_BOSSERHOF_CLASSES)) != len(LLM_BOSSERHOF_CLASSES):
    raise AssertionError('a Bosserhof class string appears twice - the answer would be ambiguous')

_conf_block = LLM_SYSTEM_PROMPT.rsplit('confidence:', 1)[1]
_conf_in_prompt = tuple(re.findall(r'^- (\w+)\s+→', _conf_block, re.M))
if _conf_in_prompt != LLM_CONFIDENCE_LEVELS:
    raise AssertionError(f'confidence levels differ - prompt {_conf_in_prompt} vs config {LLM_CONFIDENCE_LEVELS}')

if WORK_IMPLIED_BY != set(LLM_ACTIVITY_LABELS) - {'work'}:
    raise AssertionError('WORK_IMPLIED_BY must be every label but work')
if set(LLM_OUTPUT_SCHEMA['properties']) != set(LLM_OUTPUT_SCHEMA['required']):
    raise AssertionError('every field of the answer is required')

print(f'  ok  {len(LLM_ACTIVITY_LABELS)} activity labels, {len(LLM_BOSSERHOF_HEADLINES)} Bosserhof headlines with '
      f'{len(LLM_BOSSERHOF_CLASSES) - len(LLM_BOSSERHOF_HEADLINES)} subcategories ({len(LLM_BOSSERHOF_CLASSES)} valid class strings), '
      f'{len(LLM_CONFIDENCE_LEVELS)} confidence levels - prompt and config agree')
print(f'  ok  work by rule for any of: {", ".join(sorted(WORK_IMPLIED_BY))}')
print(f'  ..  prompt: {len(LLM_SYSTEM_PROMPT):,} characters, {len(LLM_SYSTEM_PROMPT.split()):,} words '
      f'(the previous pipeline\'s: 12,251 characters); a record adds a median {int(records["n_chars"].median())} characters')
print()
print('  ..  the answer schema (every reply is validated against it in 05.5):')
print(json.dumps(LLM_OUTPUT_SCHEMA, indent=2, ensure_ascii=False))

You are a building activity interpreter and classifier.

You receive ONE building per message as a short labelled record. Every record
describes a building in which human activities take place; most buildings
have one primary use, some have several. Each line names its source; an
absent line means that source has nothing to say. Read the lines with the
weight given here:

  inside:        the businesses, institutions and facilities located inside
                 the building, each as its name followed by its OpenStreetMap
                 tag in brackets, written key=value. When several share one
                 tag, the tag is given once with the count and the names after
                 it. The word unnamed stands where the mapper gave no name; it
                 is not a name. These are the actual occupants: the strongest
                 evidence for what happens in the building.
  site:          the larger complex, campus or estate the building stands in,
                 as n

## 4. Routing — who is asked

One call per building wherever the record carries evidence of its own: a POI,
a site, a name, an activity tag — 33,734 buildings. The 6,052 `class_only`
buildings carry nothing but a register class, a footprint type (`yes` or none),
the land use and a size. Two such buildings with the same class, land and size
band are the same question, and asking it twice can only produce two answers.

They are therefore grouped into **signatures** — class, footprint type, ALKIS
land use and its kind, OSM land use, footprint-area band, height band — and each
signature is asked once, on the record of its median-area member. In the
assembly (05.6) the answer is copied to every member and marked
`route = signature`, so it stays distinguishable from an answer the building got
on its own. The bands (`LLM_SIGNATURE_AREA_BINS_M2`, `LLM_SIGNATURE_HEIGHT_BINS_M`
in config) keep the size the prompt uses for scale: within one band the
footprint varies by at most a factor of two and the height by one storey band.

The saving is modest, about one call in eight. The point is consistency:
identical evidence, identical answer, by construction rather than by hoping the
model repeats itself.

In [5]:
plan = build_plan(bld, records[['building_id', 'evidence']])
require_unique(plan, 'building_id', 'plan')
if len(plan) != len(bld):
    raise AssertionError(f'the plan has {len(plan):,} rows for {len(bld):,} buildings')
_calls = plan['answered_by'].nunique()
_sig_rows = plan['route'] == 'signature'
print(f'  ok  {len(plan):,} buildings -> {_calls:,} calls: '
      f"{int((~_sig_rows).sum()):,} buildings asked on their own, "
      f"{int(_sig_rows.sum()):,} class-only buildings in {plan.loc[_sig_rows, 'signature'].nunique():,} signatures "
      f'({len(plan) - _calls:,} calls saved)')
if set(plan.loc[_sig_rows, 'evidence']) != {'class_only'} or (plan.loc[~_sig_rows, 'evidence'] == 'class_only').any():
    raise AssertionError('only and all class_only buildings may share a call')

_sig = (plan[_sig_rows].groupby('signature')
        .agg(buildings=('building_id', 'size'), answered_by=('answered_by', 'first'))
        .sort_values('buildings', ascending=False))
print(f'  ..  signature sizes: {int((_sig["buildings"] == 1).sum()):,} with one building, '
      f'{int((_sig["buildings"] >= 10).sum()):,} with ten or more, the largest {int(_sig["buildings"].max()):,}')
print()
print('  ..  the eight largest signatures:')
with pd.option_context('display.max_colwidth', 140):
    print(_sig.head(8).to_string())
print()
print('  ..  the record sent for the largest signature - its median-area member - and for one signature of a single building:')
_rec_by_id = records.set_index('building_id')['record']
print(_rec_by_id[_sig['answered_by'].iloc[0]])
print('-' * 60)
print(_rec_by_id[_sig[_sig['buildings'] == 1]['answered_by'].iloc[0]])

plan.to_parquet(LLM_PLAN_FILE, index=False)
print()
print(f'  ok  -> {LLM_PLAN_FILE.name} ({len(plan):,} rows, {LLM_PLAN_FILE.stat().st_size / 1e3:,.0f} kB); columns {list(plan.columns)}')

  ok  plan.building_id: unique and non-null (39,786)
  ok  39,786 buildings -> 34,993 calls: 33,734 buildings asked on their own, 6,052 class-only buildings in 1,259 signatures (4,793 calls saved)
  ..  signature sizes: 808 with one building, 95 with ten or more, the largest 359

  ..  the eight largest signatures:
                                                                                                                                    buildings       answered_by
signature                                                                                                                                                      
Residential buildings with trade and services | footprint yes | land commercial services | - | osm residential | <250 m2 | 10-20 m        359  DENIAL0600001y9f
Residential buildings with trade and services | footprint yes | land commercial services | - | osm residential | <250 m2 | 5-10 m         219  DENIAL03000048Jp
Buildings for trade and services | footprin

## 5. The calls

The transport is the previous pipeline's: one stateless POST per building to the
TU Braunschweig KI-Toolbox, the system prompt as `customInstructions` and the
building's record as the prompt — nothing else. The endpoint has no reasoning or
temperature setting, and buildings cannot influence each other. Every building
is asked once. Three things are different from the previous pipeline, all in
`lib/llm_client.py` and `lib/llm_run.py`:

* **Every reply is checked the moment it arrives** — parsed as JSON, validated
  against `LLM_OUTPUT_SCHEMA` — and the building is asked again at once when the
  check fails. A transport failure waits with a growing pause and sends the same
  text again; an invalid reply (no JSON, `Public facilities / schools`, an empty
  label list) is re-asked with the validation error appended to the record, so
  the model sees what was wrong. After `LLM_MAX_ATTEMPTS` the building is written
  as failed with its error and the raw reply — never as a guess. The previous
  pipeline left failures for later sweeps and lost a row when a healthy HTTP 200
  carried a malformed body.
* **Every answer is on disk the moment it is valid.** One JSON line per
  building, flushed and synced, with the prompt's hash. A crash loses at most the
  call in flight; a re-run skips every valid answer under the current prompt and
  asks everything else again — the failed ones included, without anyone having
  to find them. Answers given under an earlier prompt are never mixed in.
* **Progress you can watch**, for a machine that runs for days. In a terminal
  one line is redrawn after every call: bar, done of total, seconds per call,
  ETA with the finishing time, failures, the last answer. Every minute a status
  block goes to the log and, as JSON, to `05_llm_status.json` for a second
  terminal, with the confidence and class mix so far.

**The sample first.** Before anything larger runs, the ten dry-run buildings go
to the real model once, into their own answers file: the payload, the parsing,
the validation, the checkpoint and the progress are exercised end to end, and
the seconds per call set the runtime of the full run. A one-off check on
2026-09-14 asked the same ten three times each: labels identical on 6 of 10
(work aside), class on 6 of 10, confidence on 8 of 10, at 7.9 s per call —
consistent wherever the record carries evidence, split on the thin records where
the data supports two readings. The run itself asks every building once.

The first cell below makes no call: it checks the token is there and exercises
the validator on replies a model could give.

In [6]:
read_token()
print('  ok  API token found in .env / environment (not shown)')
print(f'  ..  endpoint {LLM_API_URL}')
print(f'  ..  model {LLM_MODEL} | the system prompt and one record per call, nothing else | up to {LLM_MAX_ATTEMPTS} attempts per building | prompt {prompt_sha()}')

# the validator on replies a model could give - no call is made here
_good = ('{"interpreted_type": "a small shop", "mid_labels": ["retail_daily"], '
         '"bosserhof_class": "retail (small-scale)", "confidence": "high", "reason": "the inside line"}')
_a = validate_answer(parse_answer('```json\n' + _good + '\n```'))
if _a['bosserhof_class'] != 'retail (small-scale)' or _a['mid_labels'] != ['retail_daily']:
    raise AssertionError(_a)
if validate_answer(parse_answer(_good.replace('"high"', '"High"')))['confidence'] != 'high':
    raise AssertionError('letter case is normalised, nothing else')
print('  ok  a valid reply passes, also inside a code fence; letter case and whitespace are normalised')
for _bad, _why in [
    (_good.replace('retail (small-scale)', 'Retail / retail (small-scale)'), 'headline and subcategory combined'),
    (_good.replace('["retail_daily"]', '[]'), 'empty label list'),
    (_good.replace('"confidence": "high", ', ''), 'a field missing'),
    (_good.replace('retail_daily', 'shopping'), 'a label outside the twelve'),
    ('The building is probably a shop.', 'no JSON at all'),
]:
    try:
        validate_answer(parse_answer(_bad))
        raise AssertionError(f'accepted a reply it must reject: {_why}')
    except InvalidAnswer as _e:
        print(f'  ok  rejected ({_why}): {_e}')

  ok  API token found in .env / environment (not shown)
  ..  endpoint https://ki-toolbox.tu-braunschweig.de/api/v1/chat/send
  ..  model gpt-oss-120b | the system prompt and one record per call, nothing else | up to 4 attempts per building | prompt 795433ed9feb
  ok  a valid reply passes, also inside a code fence; letter case and whitespace are normalised
  ok  rejected (headline and subcategory combined): bosserhof_class "Retail / retail (small-scale)" is not one of the allowed strings
  ok  rejected (empty label list): mid_labels is empty; every building receives at least one label
  ok  rejected (a field missing): missing field(s): confidence
  ok  rejected (a label outside the twelve): activity label "shopping" is not one of the allowed strings
  ok  rejected (no JSON at all): no JSON object in the reply


In [7]:
_rec = dict(zip(records['building_id'], records['record']))
_absent = [b for b in LLM_SAMPLE_BUILDING_IDS if b not in _rec]
if _absent:
    raise AssertionError(f'sample buildings without a record: {_absent}')

# resumable: a second execution of this cell asks only what has no valid answer yet
_summary = run(LLM_SAMPLE_BUILDING_IDS, _rec, LLM_SAMPLE_ANSWERS_FILE, label='sample', status_every_s=30)

sample = valid_answers(load_answers(LLM_SAMPLE_ANSWERS_FILE), prompt_sha())
sample = sample[sample['building_id'].isin(LLM_SAMPLE_BUILDING_IDS)]
if len(sample) < len(LLM_SAMPLE_BUILDING_IDS):
    raise AssertionError(f'{len(LLM_SAMPLE_BUILDING_IDS) - len(sample)} sample building(s) have no valid answer - '
                         'run this cell again to resume')

print()
print(f'=== the real model on the {len(sample)} sample buildings, once each:')
print(f"  {sample['elapsed_s'].median():.1f} s per call (median), {sample['elapsed_s'].mean():.1f} mean, "
      f"{sample['elapsed_s'].min():.1f} min, {sample['elapsed_s'].max():.1f} max; "
      f"{int((sample['attempts'] > 1).sum())} of {len(sample)} calls needed a retry")
print(f"  confidence {sample['confidence'].value_counts().to_dict()} | "
      f"work as the only label on {sum(list(l) == ['work'] for l in sample['mid_labels'])} building(s)")
for _b in LLM_SAMPLE_BUILDING_IDS:
    _a = sample[sample['building_id'] == _b].iloc[0]
    print()
    print(f"--- {_b}  |  {_rec[_b].splitlines()[0][:90]}")
    print(f"   {', '.join(_a['mid_labels']):48s} | {_a['bosserhof_class']:42s} | {_a['confidence']:6s} | {_a['elapsed_s']:5.1f}s")
    print(f"   {_a['interpreted_type'][:110]}")
    print(f"   reason: {_a['reason'][:240]}")

_plan_calls = pd.read_parquet(LLM_PLAN_FILE)['answered_by'].nunique()
_med = sample['elapsed_s'].median()
print()
print(f'  ..  at the median {_med:.1f} s per call, the {_plan_calls:,} calls of the full run take '
      f'{_plan_calls * _med / 3600:,.0f} hours = {_plan_calls * _med / 86400:.1f} days with one request at a time')

sample: 10 buildings, 10 already answered under prompt 795433ed9feb, 0 to do; 50 earlier answers under another prompt are ignored | model gpt-oss-120b, 1 request(s) at a time -> 05_llm_sample_answers.jsonl



=== the real model on the 10 sample buildings, once each:
  9.1 s per call (median), 10.0 mean, 5.7 min, 16.4 max; 0 of 10 calls needed a retry
  confidence {'high': 5, 'medium': 5} | work as the only label on 2 building(s)

--- DENIAL01000051rg  |  inside: unnamed (sport=multi)
   school, sports                                   | schools                                    | high   |   5.7s
   multi-sports hall at the Wilhelm-Gymnasium Leonhardstraße campus
   reason: The inside line (sport=multi) indicates a multi-sports facility (sports). The site name Wilhelm-Gymnasium Abt. Leonhardstraße and cadastre as a general education school show it belongs to a school (school). Hence Bosserhof class is schools.

--- DENIAL0500001k2b  |  cadastre: Buildings for public purposes, named "Gemeindehaus"
   errands, meetup                                  | normal office                              | medium |   9.5s
   Municipal community house (Gemeindehaus) for public administration and gatheri


  ..  at the median 9.1 s per call, the 34,993 calls of the full run take 89 hours = 3.7 days with one request at a time


### Launching the full run

The full run does not belong in a notebook cell: it runs for days on a machine
that stays on. `scripts/05_run_llm.py` uses the same `run` as the cell above, on
the plan from section 4, asks every planned building once, and continues where
it stopped whenever it is started again:

```bash
cd <repo>
python scripts/05_run_llm.py --sample          # the ten sample buildings once, to see it work
python scripts/05_run_llm.py --limit 200       # the first two hundred pending calls, then stop
python scripts/05_run_llm.py                   # everything
```

In a terminal — tmux or screen on the machine — one progress line is redrawn
after every call. Detached, the log gets a status block every minute:

```bash
nohup python scripts/05_run_llm.py > data/output/05_llm_run.log 2>&1 &
tail -f data/output/05_llm_run.log        # the status block, every minute
cat  data/output/05_llm_status.json       # the same facts as JSON, for a second terminal
```

`--workers N` sends N requests at a time — the KI-Toolbox rate limit is
undocumented, so raise it only after asking the operators. The cell below reads
whatever the full run has produced so far, and says so if it has not started.

In [8]:
if LLM_ANSWERS_FILE.exists():
    _full = load_answers(LLM_ANSWERS_FILE)
    _plan = pd.read_parquet(LLM_PLAN_FILE)
    _ok = _full[(_full['ok'] == True) & (_full['prompt_sha'] == prompt_sha())]
    _stale = int(((_full['ok'] == True) & (_full['prompt_sha'] != prompt_sha())).sum())
    print(f"full run: {_ok['building_id'].nunique():,} of {_plan['answered_by'].nunique():,} calls answered under the current prompt; "
          f"{int((_full['ok'] == False).sum()):,} failed line(s)" + (f'; {_stale:,} answers under an earlier prompt' if _stale else ''))
    if LLM_STATUS_FILE.exists():
        print(LLM_STATUS_FILE.read_text(encoding='utf-8')[:2000])
else:
    print('the full run has not started - launch it with scripts/05_run_llm.py as shown above')

the full run has not started - launch it with scripts/05_run_llm.py as shown above


## Where this leaves us

Sections 1 to 5 are done. The columns have a role, every building has a record,
the prompt sits in config next to the lists and the schema it is checked
against, the plan says who is asked, and the client asks, checks and re-asks.
The ten sample buildings have been classified by the real model; the answers
and the seconds per call are printed above. The full run has not been started:
it is `scripts/05_run_llm.py` on the machine that stays on — one call per
planned building, resumable, with live progress.

### Next

**05.6, assembly and validation.** The answers joined to the buildings; every
signature's answer copied to its members and marked; `work` added by rule where
any other label is present, with `work_from` saying whether the model, the rule
or both put it there; then the comparison against the rule baseline in
`activities` and the annotated Bosserhof set of the previous pipeline.